# Validation Test Evaluation - Oscillating Droplet

This notebook and the corresponding simulation setup notebook (DropletOscillating_Run.ipynb) are part of published results (cf. 7.2.2) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_Droplet");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\XNSEpaper_Droplet");

In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh10	6/8/2020 9:57:27 AM	1b80d7a7...
#1: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh60	6/8/2020 9:56:47 AM	4cd1809e...
#2: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh20	6/8/2020 9:57:11 AM	7c6fe495...
#3: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh60	5/17/2020 9:40:06 PM	1591dbb6...
#4: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh80*	5/17/2020 9:40:18 PM	c95b2833...
#5: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh40	6/6/2020 10:43:45 PM	7c07123e...
#6: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh80	6/8/2020 9:56:33 AM	f29869b3...
#7: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh80_restart1*	6/6/2020 9:55:26 PM	74912a6d...
#8: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh40	5/17/2020 9:40:01 PM	3c3518f2...
#9: XNSEpaper_Droplet	DropletInEquilibrium_meshStudy_mesh20	5/17/2020 9:39:56 PM	a7b8c038...


In [5]:
int[] Resolutions = { 10, 20, 40, 60, 80 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_Droplet");

In [7]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int Res in Resolutions) {
    string studyName = $"DropletOscillation_meshStudy_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
evalSess

#0: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh10	6/8/2020 9:57:27 AM	1b80d7a7...
#1: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh20	6/8/2020 9:57:11 AM	7c6fe495...
#2: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh40	6/6/2020 10:43:45 PM	7c07123e...
#3: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh60	6/8/2020 9:56:47 AM	4cd1809e...
#4: XNSEpaper_Droplet	DropletOscillation_meshStudy_mesh80	6/8/2020 9:56:33 AM	f29869b3...


In [9]:
NUnit.Framework.Assert.AreEqual(5, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [10]:
bool calcComputeTimes = false;

In [11]:
if (calcComputeTimes) {

foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

# temporal evaluation of semiaxis and energies

In [12]:
var evalDataDrop = evalSess.ReadLogDataForXNSE(Dropletlike.LogfileName);

In [13]:
public static Plot2Ddata GetSemiAxisOverTime_Plot2Ddata(List<Plot2Ddata> data, double xMin = 0.0, double xMax = 1000.0, double yMin = 0.4, double yMax = 0.65) {

    Plot2Ddata plt =  new Plot2Ddata();
    int index = 0;
    plt.Xlabel = "Time";
    plt.Ylabel = "Semi-Axis";
    
    var datGroups = data.ElementAt(index).dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        plt.AddDataGroup(nameData.Last(), datGroups[i].Abscissas, datGroups[i].Values, fmt);
        lcIdx++;
    }

    plt.XrangeMin = xMin;
    plt.XrangeMax = xMax;
    plt.YrangeMin = yMin;
    plt.YrangeMax = yMax;
    plt.ShowLegend = true;

    return plt;
}

In [14]:
var sess = evalSess.Pick(0);
string path = Path.Combine(sess.Database.Path, "sessions", sess.ID.ToString(), "Energy.txt");
string[] lines = File.ReadAllLines(path);
string[] values = lines[0].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries);

In [15]:
var evalDataEnergy = evalSess.ReadLogData("Energy", values);

In [16]:
public static Plot2Ddata GetEnergy_Plot2Ddata(List<Plot2Ddata> data, string meshSize) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = "Energy";

    int lcIdx = 1;

    int datGrpsCount = data.ElementAt(0).dataGroups.Count();
    for (int i = 0; i < datGrpsCount; i++) {

        int index = -1;
        for (int j = 0; j < data.Count(); j++) {
            if (data.ElementAt(j).Ylabel.ToLower().Contains($"kineticenergy") && !data.ElementAt(j).Ylabel.ToLower().Contains($"changerate")) {
                index = j;
                break;
            }
        }
        if (index == -1) {
            continue;
        }
        var datGroups = data.ElementAt(index).dataGroups;

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData[2].Contains(meshSize)) {
            plt.AddDataGroup("KineticEnergy", datGroups[i].Abscissas, datGroups[i].Values, fmt);
            lcIdx++;
        }
    }

    // Surface Energy
    double sigma = 0.1;
    double r0 = 0.25;
    double EsurfMin = sigma * 2 * Math.PI * r0; 

    for (int i = 0; i < datGrpsCount; i++) {
        
        int index = -1;
        for (int j = 0; j < data.Count(); j++) {
            if (data.ElementAt(j).Ylabel.ToLower().Contains($"surfaceenergy")) {
                index = j;
                break;
            }
        }
        if (index == -1) {
            continue;
        }
        var datGroups = data.ElementAt(index).dataGroups;

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData[2].Contains(meshSize)) {
            plt.AddDataGroup("SurfaceEnergy", datGroups[i].Abscissas, datGroups[i].Values.Select(val => val - EsurfMin).ToArray(), fmt);
            lcIdx++;
        }
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 1000.0;
    plt.ShowLegend = true;

    return plt;

} 

## FIGURE 6 

In [17]:
Plot2Ddata[,] Fig6 = new Plot2Ddata[1, 3];

Fig6[0, 0] = GetSemiAxisOverTime_Plot2Ddata(evalDataDrop);
Fig6[0, 1] = GetSemiAxisOverTime_Plot2Ddata(evalDataDrop, 800.0, 1000.0, 0.4997, 0.5003);
Fig6[0, 2] = GetEnergy_Plot2Ddata(evalDataEnergy, "mesh80");

Fig6.ToGnuplot().PlotSVG(xRes:2000,yRes:500)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 0 
 
 
 
 
 200 
 
 
 
 
 400 
 
 
 
 
 600 
 
 
 
 
 800 
 
 
 
 
 1000 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Semi-Axis 
 
 
 
 
 Time 
 
 
 
 
 mesh10 
 
 
 
 
 mesh10 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M470.7,62.1 L524.1,62.1 M86.8,80.5 L87.8,71.2 L88.9,70.1 L89.9,73.0 L91.0,78.3 L92.0,85.1
 L93.1,93.0 L94.1,101.7 L95.2,111.2 L96.2,121.1 L97.3,131.4 L98.3,141.9 L99.4,152.6 L100.4,163.3
 L101.5,173.9 L102.5,184.5 L103.5,196.7 L104.6,206.8 L105.6,216.7 L106.7,226.5 L107.7,236.1 L108.8,245.5
 L109.8,254.7 L110.9,263.5 L111.9,272.0 L113.0,280.3 L114.0,288.3 L115.1,296.0 L116.1,303.5 L117.2,310.8
 L118.2,317.9 L119.2,324.6 L120.3,330.9 L121.3,336.7 L122.4,342.0 L123.4,346.8 L124.5,351.1 L125.5,354.8
 L126.6,357.9 L127.6,360.5 L128.7,362.6 L129.7,364.1 L130.8,365.2 L131.8,365.8 L132.9,365.9 L133.9,365.7
 L134.9,365.0 L136.0,364.0 L137.0,362.6 L138.1,360.9 L139.1,358.8 L140.2,356.4 L141.2,353.8 L142.3,350.9
 L143.3,347.8 L144.4,344.4 L145.4,340.8 L146.5,337.1 L147.5,333.2 L148.5,329.1 L149.6,324.9 L150.6,320.7
 L151.7,316.3 L152.7,311.9 L153.8,307.5 L154.8,303.0 L155.9,298.6 L156.9,294.1 L158.0,289.7 L159.0,285.5
 L160.1,281.3 L161.1,277.2 L162.2,273.3 L163.2,269.6 L164.2,266.0 L165.3,262.6 L166.3,259.5 L167.4,256.5
 L168.4,253.9 L169.5,251.5 L170.5,249.3 L171.6,247.5 L172.6,245.9 L173.7,244.5 L174.7,243.5 L175.8,242.7
 L176.8,242.2 L177.9,241.9 L178.9,241.8 L179.9,242.0 L181.0,242.4 L182.0,243.0 L183.1,243.7 L184.1,244.7
 L185.2,245.8 L186.2,247.1 L187.3,248.5 L188.3,250.1 L189.4,251.8 L190.4,253.5 L191.5,255.4 L192.5,257.4
 L193.6,259.4 L194.6,261.5 L195.6,263.6 L196.7,265.8 L197.7,267.9 L198.8,270.1 L199.8,272.3 L200.9,274.5
 L201.9,276.7 L203.0,278.8 L204.0,280.9 L205.1,282.9 L206.1,284.9 L207.2,286.8 L208.2,288.6 L209.3,290.3
 L210.3,292.0 L211.3,293.5 L212.4,294.9 L213.4,296.2 L214.5,297.4 L215.5,298.4 L216.6,299.4 L217.6,300.1
 L218.7,300.8 L219.7,301.3 L220.8,301.7 L221.8,302.0 L222.9,302.2 L223.9,302.3 L225.0,302.2 L226.0,302.1
 L227.0,301.9 L228.1,301.6 L229.1,301.2 L230.2,300.8 L231.2,300.2 L232.3,299.6 L233.3,298.9 L234.4,298.2
 L235.4,297.4 L236.5,296.6 L237.5,295.7 L238.6,294.8 L239.6,293.9 L240.7,292.9 L241.7,291.9 L242.7,290.8
 L243.8,289.8 L244.8,288.8 L245.9,287.7 L246.9,286.7 L248.0,285.7 L249.0,284.6 L250.1,283.7 L251.1,282.7
 L252.2,281.8 L253.2,280.9 L254.3,280.0 L255.3,279.2 L256.3,278.4 L257.4,277.7 L258.4,277.1 L259.5,276.5
 L260.5,275.9 L261.6,275.5 L262.6,275.1 L263.7,274.7 L264.7,274.4 L265.8,274.2 L266.8,274.0 L267.9,273.9
 L268.9,273.8 L270.0,273.8 L271.0,273.8 L272.0,273.9 L273.1,274.1 L274.1,274.2 L275.2,274.4 L276.2,274.7
 L277.3,274.9 L278.3,275.3 L279.4,275.6 L280.4,276.0 L281.5,276.4 L282.5,276.8 L283.6,277.2 L284.6,277.7
 L285.7,278.2 L286.7,278.7 L287.7,279.2 L288.8,279.7 L289.8,280.2 L290.9,280.7 L291.9,281.2 L293.0,281.7
 L294.0,282.2 L295.1,282.6 L296.1,283.1 L297.2,283.6 L298.2,284.0 L299.3,284.4 L300.3,284.8 L301.4,285.2
 L302.4,285.6 L303.4,285.9 L304.5,286.2 L305.5,286.5 L306.6,286.7 L307.6,286.9 L308.7,287.1 L309.7,287.3
 L310.8,287.4 L311.8,287.5 L312.9,287.5 L313.9,287.6 L315.0,287.6 L316.0,287.6 L317.1,287.6 L318.1,287.5
 L319.1,287.5 L320.2,287.4 L321.2,287.3 L322.3,287.1 L323.3,287.0 L324.4,286.9 L325.4,286.7 L326.5,286.5
 L327.5,286.3 L328.6,286.1 L329.6,285.9 L330.7,285.7 L331.7,285.4 L332.8,285.2 L333.8,285.0 L334.8,284.7
 L335.9,284.5 L336.9,284.2 L338.0,284.0 L339.0,283.8 L340.1,283.5 L341.1,283.3 L342.2,283.1 L343.2,282.9
 L344.3,282.7 L345.3,282.5 L346.4,282.3 L347.4,282.1 L348.4,281.9 L349.5,281.8 L350.5,281.6 L35

In [18]:
public static void EvaluateTerminalTimeStep(Plot2Ddata pltDat, double[] lowerThrshldValues, double[] upperThrshldValues) {

    int numGrps = pltDat.dataGroups.Count();
    if (lowerThrshldValues.Length != numGrps || upperThrshldValues.Length != numGrps) {
        throw new ArgumentException("Threshold value arrays must match number of data groups.");
    }

    for (int i = 0; i < numGrps; i++) {
        var grp = pltDat.dataGroups[i];
        // last time step index
        int timeIdx  = grp.Abscissas.Length - 1;
        double time = grp.Abscissas[timeIdx];

        double valAtTime = grp.Values[timeIdx];
        Console.WriteLine($"Data group: {grp.Name}, value at time {time}: {valAtTime}");

        NUnit.Framework.Assert.IsTrue(valAtTime >= lowerThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is below lower threshold {lowerThrshldValues[i]}.");

        NUnit.Framework.Assert.IsTrue(valAtTime <= upperThrshldValues[i], 
            $"Value {valAtTime} of data group {grp.Name} at time {time} is above upper threshold {upperThrshldValues[i]}.");
    }
}

In [21]:
EvaluateTerminalTimeStep(Fig6[0, 0], 
    new double[] { 0.4970, 0.4990, 0.4990, 0.4990, 0.4990 }, 
    new double[] { 0.5010, 0.5010, 0.5010, 0.5010, 0.5010 });

In [22]:
EvaluateTerminalTimeStep(Fig6[0, 2], new double[] { 0.0, 0.0 }, new double[] { 1e-8, 1e-4 });